In [ ]:

# ============================================================
# 0. LOAD AMAZON DATASET
# ============================================================

print("=" * 70)
print("LOADING AMAZON DATA")
print("=" * 70)

base = "/kaggle/input/datasets/shreyamarde/amazon-ml/student_resource/"
train_path = base + "/dataset/train"

train1 = pd.read_csv(
    train_path + "/train_source1.tsv",
    sep="\t"
)

train2 = pd.read_csv(
    train_path + "/train_source2.tsv",
    sep="\t"
)

train3 = pd.read_csv(
    train_path + "/train_source3.tsv",
    sep="\t"
)

gt = pd.read_csv(
    train_path + "/train_ground_truth.tsv",
    sep="\t"
)

print("train1:", train1.shape)
print("train2:", train2.shape)
print("train3:", train3.shape)
print("gt    :", gt.shape)


# ============================================================
# 1. SETTINGS
# ============================================================

# Start with 2,000 S1 entities.
# Once everything works, we will scale this.
N_S1 = 2000

# Random same-country candidates considered per source.
NEGATIVE_POOL_PER_SOURCE = 100

# Keep hardest wrong candidates.
HARD_NEGATIVES_PER_S1 = 5

RANDOM_STATE = 42


# ============================================================
# 2. NORMALIZATION
# ============================================================

ADDRESS_ABBREVIATIONS = {
    "street": "st",
    "road": "rd",
    "avenue": "ave",
    "boulevard": "blvd",
    "lane": "ln",
    "drive": "dr",
    "highway": "hwy",
    "parkway": "pkwy",
    "place": "pl",
    "court": "ct",
    "circle": "cir",
    "terrace": "ter",
    "square": "sq",
}


def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    text = re.sub(
        r"[^a-z0-9\u0900-\u097f\u0b80-\u0bff]+",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def normalize_name(text):
    return normalize_text(text)


def normalize_address(text):

    text = normalize_text(text)

    if not text:
        return ""

    words = text.split()

    words = [
        ADDRESS_ABBREVIATIONS.get(
            word,
            word
        )
        for word in words
    ]

    return " ".join(words)


# ============================================================
# 3. SIMILARITY FUNCTIONS
# ============================================================

def word_jaccard(text1, text2):

    if not text1 or not text2:
        return 0.0

    set1 = set(text1.split())
    set2 = set(text2.split())

    if not set1 or not set2:
        return 0.0

    return len(
        set1 & set2
    ) / len(
        set1 | set2
    )


def char_ngrams(text, n=3):

    if not text:
        return set()

    text = text.replace(
        " ",
        ""
    )

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(
            len(text) - n + 1
        )
    }


def char_jaccard(text1, text2, n=3):

    grams1 = char_ngrams(
        text1,
        n
    )

    grams2 = char_ngrams(
        text2,
        n
    )

    if not grams1 or not grams2:
        return 0.0

    return len(
        grams1 & grams2
    ) / len(
        grams1 | grams2
    )


# ============================================================
# 4. SAMPLE S1 ENTITIES
# ============================================================

print()
print("=" * 70)
print("PREPARING S1 SAMPLE")
print("=" * 70)

s1_sample = train1.sample(
    n=min(
        N_S1,
        len(train1)
    ),
    random_state=RANDOM_STATE
).copy()

s1_sample["name_norm"] = (
    s1_sample["business_name"]
    .fillna("")
    .apply(normalize_name)
)

s1_sample["address_norm"] = (
    s1_sample["business_address"]
    .fillna("")
    .apply(normalize_address)
)

print(
    "S1 sample:",
    len(s1_sample)
)


# ============================================================
# 5. GET GROUND TRUTH FOR SAMPLE
# ============================================================

sample_s1_ids = set(
    s1_sample["entity_id"]
)

sample_gt = gt[
    gt["source1_entity_id"].isin(
        sample_s1_ids
    )
].copy()

print(
    "Ground-truth S1 rows:",
    len(sample_gt)
)


# ============================================================
# 6. CREATE POSITIVE PAIRS
# ============================================================

positive_pairs = []

for _, row in sample_gt.iterrows():

    s1_id = row[
        "source1_entity_id"
    ]

    matched = row[
        "matched_entity_ids"
    ]

    if pd.isna(matched):
        continue

    matched = str(
        matched
    ).strip()

    if not matched:
        continue

    for candidate_id in matched.split(","):

        candidate_id = candidate_id.strip()

        if candidate_id:

            positive_pairs.append(
                (
                    s1_id,
                    candidate_id
                )
            )


positive_pairs = list(
    set(positive_pairs)
)

positive_set = set(
    positive_pairs
)

print(
    "Positive pairs:",
    len(positive_pairs)
)


# ============================================================
# 7. GET POSITIVE S2/S3 RECORDS
# ============================================================

positive_s2_ids = [
    cid
    for _, cid in positive_pairs
    if cid.startswith("S2-")
]

positive_s3_ids = [
    cid
    for _, cid in positive_pairs
    if cid.startswith("S3-")
]

positive_s2 = train2[
    train2["entity_id"].isin(
        positive_s2_ids
    )
].copy()

positive_s3 = train3[
    train3["entity_id"].isin(
        positive_s3_ids
    )
].copy()


# ============================================================
# 8. CREATE LOOKUPS FOR POSITIVE RECORDS
# ============================================================

s2_lookup = {
    row["entity_id"]: row
    for _, row in positive_s2.iterrows()
}

s3_lookup = {
    row["entity_id"]: row
    for _, row in positive_s3.iterrows()
}


# ============================================================
# 9. RANDOM GENERATOR
# ============================================================

rng = np.random.default_rng(
    RANDOM_STATE
)


def sample_country_records(
    source_df,
    country,
    n
):

    pool = source_df[
        source_df["country"] == country
    ]

    if len(pool) <= n:
        return pool.copy()

    indices = rng.choice(
        len(pool),
        size=n,
        replace=False
    )

    return pool.iloc[
        indices
    ].copy()


# ============================================================
# 10. FEATURE FUNCTION
# ============================================================

def make_features(
    s1_row,
    candidate_row
):

    name1 = s1_row[
        "name_norm"
    ]

    addr1 = s1_row[
        "address_norm"
    ]

    name2 = normalize_name(
        candidate_row[
            "business_name"
        ]
    )

    addr2 = normalize_address(
        candidate_row[
            "business_address"
        ]
    )

    name_word = word_jaccard(
        name1,
        name2
    )

    address_word = word_jaccard(
        addr1,
        addr2
    )

    name_char = char_jaccard(
        name1,
        name2
    )

    address_char = char_jaccard(
        addr1,
        addr2
    )

    if max(
        len(name1),
        len(name2)
    ) > 0:

        name_length_ratio = (
            min(
                len(name1),
                len(name2)
            )
            /
            max(
                len(name1),
                len(name2)
            )
        )

    else:

        name_length_ratio = 0.0


    if max(
        len(addr1),
        len(addr2)
    ) > 0:

        address_length_ratio = (
            min(
                len(addr1),
                len(addr2)
            )
            /
            max(
                len(addr1),
                len(addr2)
            )
        )

    else:

        address_length_ratio = 0.0


    max_similarity = max(
        name_word,
        address_word,
        name_char,
        address_char
    )

    average_similarity = (
        name_word
        + address_word
        + name_char
        + address_char
    ) / 4


    return {

        "name_word": name_word,

        "address_word": address_word,

        "name_char": name_char,

        "address_char": address_char,

        "name_length_ratio":
            name_length_ratio,

        "address_length_ratio":
            address_length_ratio,

        "max_similarity":
            max_similarity,

        "average_similarity":
            average_similarity
    }


# ============================================================
# 11. CREATE TRAINING PAIRS
# ============================================================

print()
print("=" * 70)
print("CREATING POSITIVE + HARD NEGATIVE PAIRS")
print("=" * 70)

training_rows = []


# ------------------------------------------------------------
# POSITIVE PAIRS
# ------------------------------------------------------------

for _, s1 in s1_sample.iterrows():

    s1_id = s1[
        "entity_id"
    ]

    matches = [
        cid
        for sid, cid
        in positive_pairs
        if sid == s1_id
    ]

    for cid in matches:

        if cid.startswith("S2-"):

            candidate = s2_lookup.get(
                cid
            )

        else:

            candidate = s3_lookup.get(
                cid
            )

        if candidate is None:
            continue

        features = make_features(
            s1,
            candidate
        )

        features["s1_id"] = s1_id
        features["candidate_id"] = cid
        features["label"] = 1

        training_rows.append(
            features
        )


# ------------------------------------------------------------
# NEGATIVE PAIRS
# ------------------------------------------------------------

for counter, (_, s1) in enumerate(
    s1_sample.iterrows()
):

    s1_id = s1[
        "entity_id"
    ]

    country = s1[
        "country"
    ]


    # Random same-country candidates
    neg_s2 = sample_country_records(
        train2,
        country,
        NEGATIVE_POOL_PER_SOURCE
    )

    neg_s3 = sample_country_records(
        train3,
        country,
        NEGATIVE_POOL_PER_SOURCE
    )


    candidates = pd.concat(
        [
            neg_s2,
            neg_s3
        ],
        ignore_index=True
    )


    scored_negatives = []


    for _, candidate in candidates.iterrows():

        candidate_id = candidate[
            "entity_id"
        ]

        # Never label a real match as negative.
        if (
            s1_id,
            candidate_id
        ) in positive_set:

            continue


        features = make_features(
            s1,
            candidate
        )


        scored_negatives.append(
            (
                features[
                    "max_similarity"
                ],
                candidate_id,
                features
            )
        )


    # Highest similarity = hardest negative
    scored_negatives.sort(
        key=lambda x: x[0],
        reverse=True
    )


    hard_negatives = scored_negatives[
        :HARD_NEGATIVES_PER_S1
    ]


    for _, candidate_id, features in hard_negatives:

        features["s1_id"] = s1_id
        features["candidate_id"] = candidate_id
        features["label"] = 0

        training_rows.append(
            features
        )


    if (
        counter + 1
    ) % 500 == 0:

        print(
            f"Processed "
            f"{counter + 1:,}/"
            f"{len(s1_sample):,} S1s"
        )


pairs_df = pd.DataFrame(
    training_rows
)


print()
print(
    "Training dataset shape:",
    pairs_df.shape
)

print()
print(
    "Labels:"
)

print(
    pairs_df[
        "label"
    ].value_counts()
)


# ============================================================
# 12. SPLIT BY S1 ENTITY
# ============================================================

print()
print("=" * 70)
print("TRAIN / VALIDATION SPLIT")
print("=" * 70)

unique_s1 = pairs_df[
    "s1_id"
].unique()


train_s1_ids, val_s1_ids = train_test_split(
    unique_s1,
    test_size=0.20,
    random_state=RANDOM_STATE
)


train_pairs = pairs_df[
    pairs_df[
        "s1_id"
    ].isin(train_s1_ids)
].copy()


val_pairs = pairs_df[
    pairs_df[
        "s1_id"
    ].isin(val_s1_ids)
].copy()


print(
    "Train S1:",
    len(train_s1_ids)
)

print(
    "Validation S1:",
    len(val_s1_ids)
)

print(
    "Train pairs:",
    len(train_pairs)
)

print(
    "Validation pairs:",
    len(val_pairs)
)


# ============================================================
# 13. FEATURES
# ============================================================

feature_columns = [

    "name_word",

    "address_word",

    "name_char",

    "address_char",

    "name_length_ratio",

    "address_length_ratio",

    "max_similarity",

    "average_similarity"

]


X_train = train_pairs[
    feature_columns
]

y_train = train_pairs[
    "label"
]

X_val = val_pairs[
    feature_columns
]

y_val = val_pairs[
    "label"
]


# ============================================================
# 14. TRAIN MODEL
# ============================================================

print()
print("=" * 70)
print("TRAINING ML MODEL")
print("=" * 70)


model = HistGradientBoostingClassifier(

    max_iter=200,

    learning_rate=0.08,

    max_leaf_nodes=15,

    l2_regularization=1.0,

    random_state=RANDOM_STATE

)


model.fit(
    X_train,
    y_train
)


print(
    "Model training complete."
)


# ============================================================
# 15. VALIDATION PROBABILITIES
# ============================================================

val_probabilities = model.predict_proba(
    X_val
)[:, 1]


# ============================================================
# 16. TEST DIFFERENT FINAL THRESHOLDS
# ============================================================

print()
print("=" * 70)
print("TUNING FINAL MODEL THRESHOLD")
print("=" * 70)


threshold_results = []


thresholds = np.arange(
    0.10,
    0.96,
    0.05
)


for threshold in thresholds:

    predictions = (
        val_probabilities
        >= threshold
    ).astype(int)


    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )


    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )


    f05 = fbeta_score(
        y_val,
        predictions,
        beta=0.5,
        zero_division=0
    )


    threshold_results.append({

        "threshold":
            round(
                threshold,
                2
            ),

        "precision":
            precision,

        "recall":
            recall,

        "F0.5":
            f05

    })


threshold_df = pd.DataFrame(
    threshold_results
)


print(
    threshold_df.to_string(
        index=False
    )
)


# ============================================================
# 17. BEST THRESHOLD
# ============================================================

best_row = threshold_df.loc[
    threshold_df[
        "F0.5"
    ].idxmax()
]


BEST_THRESHOLD = float(
    best_row[
        "threshold"
    ]
)


print()
print("=" * 70)
print("BEST FINAL MODEL THRESHOLD")
print("=" * 70)


print(
    f"Threshold : "
    f"{BEST_THRESHOLD:.2f}"
)

print(
    f"Precision : "
    f"{best_row['precision']:.4f}"
)

print(
    f"Recall    : "
    f"{best_row['recall']:.4f}"
)

print(
    f"F0.5      : "
    f"{best_row['F0.5']:.4f}"
)


# ============================================================
# 18. FINAL VALIDATION REPORT
# ============================================================

final_val_predictions = (
    val_probabilities
    >= BEST_THRESHOLD
).astype(int)


print()
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)


print(
    classification_report(
        y_val,
        final_val_predictions,
        digits=4,
        zero_division=0
    )
)


print()
print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)


print(
    confusion_matrix(
        y_val,
        final_val_predictions
    )
)


# ============================================================
# 19. SAVE MODEL
# ============================================================

model_package = {

    "model": model,

    "threshold":
        BEST_THRESHOLD,

    "feature_columns":
        feature_columns

}


joblib.dump(
    model_package,
    "/kaggle/working/entity_resolution_model.pkl"
)


# ============================================================
# 20. SAVE EXPERIMENT RESULTS
# ============================================================

pairs_df.to_csv(
    "/kaggle/working/training_pairs.csv",
    index=False
)


threshold_df.to_csv(
    "/kaggle/working/threshold_results.csv",
    index=False
)


print()
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print(
    "/kaggle/working/entity_resolution_model.pkl"
)

print(
    "/kaggle/working/training_pairs.csv"
)

print(
    "/kaggle/working/threshold_results.csv"
)


# ============================================================
# 21. SUMMARY
# ============================================================

print()
print("=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)

print(
    f"S1 entities used    : "
    f"{len(s1_sample):,}"
)

print(
    f"Training pairs      : "
    f"{len(train_pairs):,}"
)

print(
    f"Validation pairs    : "
    f"{len(val_pairs):,}"
)

print(
    f"Features            : "
    f"{len(feature_columns)}"
)

print(
    f"Best threshold      : "
    f"{BEST_THRESHOLD:.2f}"
)

print(
    f"Validation F0.5     : "
    f"{best_row['F0.5']:.4f}"
)

print(
    f"Validation precision : "
    f"{best_row['precision']:.4f}"
)

print(
    f"Validation recall    : "
    f"{best_row['recall']:.4f}"
)

print()
print(
    "NEXT: Scale the candidate-generation"
    " pipeline to the full dataset."
)

LOADING AMAZON DATA
train1: (2206821, 4)
train2: (5034616, 4)
train3: (5285603, 4)
gt    : (2206821, 2)

PREPARING S1 SAMPLE
S1 sample: 2000
Ground-truth S1 rows: 2000
Positive pairs: 6924

CREATING POSITIVE + HARD NEGATIVE PAIRS
Processed 500/2,000 S1s
